# 📋 Agent 4 — Report Writer
## Generates the final HTML + PDF competitive intelligence report

**What this agent does:**
- Reads `data/analysed/full_analysis.json` from Agent 3
- Generates a complete dark-theme HTML report (10 sections)
- Exports a PDF version
- Saves an executive summary Markdown file
- Opens the report in your browser

**SDK: Anthropic (Claude) — NOT OpenAI**

**Input ← Agent 3 | Output → Final Report**

## 1. Install Dependencies

In [ ]:
%pip install anthropic python-dotenv weasyprint jinja2 --quiet

## 2. Setup

In [ ]:
import os, json, webbrowser
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()
print('ANTHROPIC_API_KEY loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))

BASE_DIR      = Path('.')
DATA_ANALYSED = BASE_DIR / 'data' / 'analysed'
REPORTS_DIR   = BASE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

with open('input_schema.json') as f:
    INPUT = json.load(f)

client = anthropic.Anthropic()
print('Ready. Business:', INPUT['business']['name'])

## 3. Check Analysis from Agent 3

In [ ]:
af = DATA_ANALYSED / 'full_analysis.json'
if not af.exists():
    print('ERROR: full_analysis.json not found. Run Agent 3 first.')
else:
    ANALYSIS = json.loads(af.read_text())
    print(f'Analysis loaded ({af.stat().st_size:,} bytes)')
    print('Keys:', list(ANALYSIS.keys()))
    print('Competitors scored:', len(ANALYSIS.get('competitor_scores', [])))
    print('Summary:', str(ANALYSIS.get('executive_summary', ''))[:150])

## 4. Define Tools

In [ ]:
# ── Tool functions ──────────────────────────────────────────────

def get_full_analysis() -> str:
    """Returns the complete structured analysis from Agent 3."""
    p = DATA_ANALYSED / 'full_analysis.json'
    return p.read_text() if p.exists() else json.dumps({'error': 'No analysis found. Run Agent 3 first.'})


def get_report_template() -> str:
    """Returns HTML report structure, section list, and design guidelines."""
    business = INPUT['business']['name']
    return (
        f'Report for: {business}. '
        'Required sections: '
        '1. Header (business name, report date, tagline). '
        '2. Executive Summary (3-4 sentence strategic insight). '
        '3. Competitor Threat Scores (CSS progress bars, color-coded: red=high, amber=medium, green=low). '
        '4. Pricing Comparison Table (per-kg rates, express, subscriptions, our advantage highlighted). '
        '5. Feature Matrix (checkmark ✓ / cross ✗ grid). '
        '6. Messaging & Positioning (competitor headline cards). '
        '7. Strategic Signals (blog + hiring activity). '
        '8. Gaps & Opportunities (color-coded cards). '
        '9. Top 3 Recommendations (numbered, specific, bold). '
        '10. Footer (generated date, confidential label). '
        'Design: dark theme bg #060d1f, cards #0f172a, accent #818cf8, text #e2e8f0. '
        'Inline CSS only — no external files. Self-contained single HTML file. '
        'Real data only — no placeholder text.'
    )


def generate_html_report(html_content: str) -> str:
    """Saves the complete HTML report to disk."""
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    p = REPORTS_DIR / f'competitive_report_{ts}.html'
    p.write_text(html_content, encoding='utf-8')
    latest = REPORTS_DIR / 'competitive_report_latest.html'
    latest.write_text(html_content, encoding='utf-8')
    print(f'  HTML saved: {p.name} ({p.stat().st_size:,} bytes)')
    return str(p)


def export_pdf_report(html_path: str) -> str:
    """Converts HTML report to PDF using WeasyPrint."""
    try:
        from weasyprint import HTML
        pdf_path = html_path.replace('.html', '.pdf')
        HTML(filename=html_path).write_pdf(pdf_path)
        size = Path(pdf_path).stat().st_size
        print(f'  PDF exported: {Path(pdf_path).name} ({size:,} bytes)')
        return pdf_path
    except Exception as e:
        msg = f'PDF skipped ({e}) — open HTML in Chrome and use Ctrl+P to save as PDF'
        print(f'  {msg}')
        return msg


def save_executive_summary(summary_markdown: str) -> str:
    """Saves the executive summary as a Markdown file."""
    p = REPORTS_DIR / 'executive_summary.md'
    p.write_text(summary_markdown, encoding='utf-8')
    print(f'  Executive summary saved: {p}')
    return str(p)


# ── Anthropic tool definitions ──────────────────────────────────
TOOLS = [
    {
        "name": "get_full_analysis",
        "description": "Returns the complete structured competitor analysis JSON from Agent 3.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "get_report_template",
        "description": "Returns the HTML report section structure and design guidelines.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "generate_html_report",
        "description": "Saves a complete self-contained HTML report to disk.",
        "input_schema": {
            "type": "object",
            "properties": {
                "html_content": {
                    "type": "string",
                    "description": "Complete HTML document as a string"
                }
            },
            "required": ["html_content"]
        }
    },
    {
        "name": "export_pdf_report",
        "description": "Converts the HTML report to a PDF file using WeasyPrint.",
        "input_schema": {
            "type": "object",
            "properties": {
                "html_path": {
                    "type": "string",
                    "description": "Absolute path to the saved HTML file"
                }
            },
            "required": ["html_path"]
        }
    },
    {
        "name": "save_executive_summary",
        "description": "Saves a concise executive summary as a Markdown file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "summary_markdown": {
                    "type": "string",
                    "description": "Markdown formatted executive summary (max 400 words)"
                }
            },
            "required": ["summary_markdown"]
        }
    },
]

# ── Tool dispatcher ─────────────────────────────────────────────
TOOL_FNS = {
    "get_full_analysis":      lambda **k: get_full_analysis(),
    "get_report_template":    lambda **k: get_report_template(),
    "generate_html_report":   lambda **k: generate_html_report(**k),
    "export_pdf_report":      lambda **k: export_pdf_report(**k),
    "save_executive_summary": lambda **k: save_executive_summary(**k),
}

print('Agent 4 tools ready:', [t['name'] for t in TOOLS])

## 5. Agentic Loop (Claude)

In [ ]:
def run_claude_agent(system: str, tools: list, tool_fns: dict, prompt: str,
                     model: str = 'claude-opus-4-8', max_tokens: int = 16000) -> str:
    """
    Runs a Claude agentic tool-use loop.
    Uses high max_tokens for Agent 4 — generates a large HTML document.
    """
    messages = [{"role": "user", "content": prompt}]
    iteration = 0

    while True:
        iteration += 1
        print(f'  [loop {iteration}] calling Claude ({model})...')

        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == 'end_turn':
            return next((b.text for b in response.content if hasattr(b, 'text')), '')

        if response.stop_reason == 'tool_use':
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    arg_keys = list((block.input or {}).keys())
                    # For html_content, just show length not full content
                    if 'html_content' in arg_keys:
                        html_len = len((block.input or {}).get('html_content', ''))
                        print(f'    → tool: {block.name}(html_content={html_len} chars)')
                    else:
                        print(f'    → tool: {block.name}({arg_keys})')
                    fn = tool_fns.get(block.name)
                    try:
                        result = fn(**(block.input or {})) if fn else f'Unknown tool: {block.name}'
                    except Exception as e:
                        result = f'Tool error: {e}'
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            return f'Unexpected stop_reason: {response.stop_reason}'

## 6. Run Agent 4

In [ ]:
SYSTEM = f"""
You are Agent 4 — the Report Writer in a competitive intelligence pipeline.
You produce the final deliverable for: {INPUT['business']['name']}.

WORKFLOW (follow in order):
1. get_full_analysis() — load all data
2. get_report_template() — get design guidelines
3. Generate a COMPLETE, self-contained HTML document with ALL 10 sections:
   Section 1  — Header with business name, report date, tagline
   Section 2  — Executive Summary (insight-rich, 3-4 sentences)
   Section 3  — Competitor Threat Scores with CSS bars (color-coded)
   Section 4  — Pricing Comparison Table (highlight our advantage)
   Section 5  — Feature Matrix (✓ / ✗ grid, all competitors)
   Section 6  — Messaging & Positioning (headline cards per competitor)
   Section 7  — Strategic Signals (blog activity + hiring)
   Section 8  — Gaps & Opportunities (color-coded cards)
   Section 9  — Top 3 Recommendations (numbered, bold, specific)
   Section 10 — Footer (date, confidential label)
4. generate_html_report(html_content) — save the HTML
5. export_pdf_report(html_path) — export PDF
6. Write concise executive summary in Markdown (max 400 words)
7. save_executive_summary(summary_markdown)

DESIGN RULES:
- Inline CSS only — no CDN, no external files
- Dark theme: background #060d1f, cards #0f172a, accent #818cf8, text #e2e8f0
- CSS progress bars for threat scores
- Green ✓ and red ✗ for feature matrix
- Real data only — zero placeholder text
- This is the final client deliverable — make it impressive
"""

PROMPT = (
    f'Generate the complete {INPUT["business"]["name"]} competitive intelligence report. '
    'All 10 sections, dark theme, real data only. '
    'Save HTML + PDF + executive summary.'
)

print('Running Agent 4 — Report Writer (claude-opus-4-8)')
print('=' * 60)
t0 = datetime.now()

output = run_claude_agent(
    system=SYSTEM,
    tools=TOOLS,
    tool_fns=TOOL_FNS,
    prompt=PROMPT,
    model='claude-opus-4-8',
    max_tokens=16000
)

elapsed = (datetime.now() - t0).seconds
print(f'\nDone in {elapsed}s')
print(output[:400] if output else '(no text output)')

## 7. Open Report & Show Summary

In [ ]:
print('Output files:')
for f in sorted(REPORTS_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size:,} bytes)')

html = REPORTS_DIR / 'competitive_report_latest.html'
if html.exists():
    webbrowser.open(f'file://{html.resolve()}')
    print('\nReport opened in browser!')
else:
    print('\nHTML report not found — check agent output above')

em = REPORTS_DIR / 'executive_summary.md'
if em.exists():
    print('\n' + '=' * 60)
    print(em.read_text())

## 🎉 Pipeline Complete!

All 4 agents ran successfully.

| File | Location |
|---|---|
| HTML Report | `reports/competitive_report_latest.html` |
| PDF Report | `reports/competitive_report_latest.pdf` |
| Executive Summary | `reports/executive_summary.md` |
| Raw Scraped Data | `data/raw/*.json` |
| Full Analysis | `data/analysed/full_analysis.json` |

To re-run individual stages: open the relevant notebook (`01`–`04`).
To run everything at once: open `05_orchestrator.ipynb` → Run All Cells.